# GigaAM Benchmark For Three Datasets

This notebook runs `GigaAM-v3` on:
- `eval_dataset_gtts`
- `eval_dataset_meetings_gtts`
- `test_dataset_v1`

Expected folders in Colab:
- `/content/datasets/eval_dataset_gtts`
- `/content/datasets/eval_dataset_meetings_gtts`
- `/content/datasets/test_dataset_v1`


In [ ]:
!pip install -q transformers==4.57.1 sentencepiece jiwer

In [ ]:
from pathlib import Path

DATASETS_ROOT = Path('/content/datasets')
RUNS = {
    'eval_dataset_gtts': DATASETS_ROOT / 'eval_dataset_gtts',
    'eval_dataset_meetings_gtts': DATASETS_ROOT / 'eval_dataset_meetings_gtts',
    'test_dataset_v1': DATASETS_ROOT / 'test_dataset_v1',
}
RESULTS_ROOT = Path('/content/gigaam_benchmark_results_all')

for name, dataset_dir in RUNS.items():
    if not dataset_dir.exists():
        raise FileNotFoundError(f'Dataset not found: {dataset_dir}')
    print(f'{name}: {dataset_dir}')

In [ ]:
import json
import os
import re
import time
from pathlib import Path
from typing import Any

import jiwer
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
MODEL_ID = 'ai-sage/GigaAM-v3'
OIL_GAS_GLOSSARY = (
    'УПН, ПСН, УПСВ, УКПГ, ГИС, ГРР, КРС, ППД, ЭЦН, АГЗУ, ГРП, ГТМ, АГНКС, АДР, '
    'АСЭЗ, ВВП, ВИЭ, ГКМ, ГМТ, ГПА, ГПЗ, ГПК, ГРС, ГТС, ГЭС, ДМС, ДО, ДТП, ESG, '
    'ЕСУПБ, ЗВ, ИТС, КМН, КПГ, КПО, КПЭ, КриоАЗС, КС, КСГЗ, КСПГ, МГ, МКС, МСП, '
    'МСФО, МТР, НГКМ, НДПИ, НДС, НДТ, НИР, НИОКР, НКО, ОМС, ООС, ООПТ, ОПО, ПГ, '
    'ПНГ, ПХГ, ПЭМ, РО, РСПП, СМИ, СПГ, СТО, СУРиВК, СЦП, СЭМ, ТЭР, ФО, ФОК, ЧС, '
    'ЭТП, НПЗ, KPI, IT, CO2, RFID.'
)
GLOSSARY_SET = {word.strip().lower() for word in OIL_GAS_GLOSSARY.replace('.', '').split(',')}
WORD_TO_NUM = {
    'ноль': '0', 'один': '1', 'одна': '1', 'два': '2', 'две': '2', 'три': '3',
    'четыре': '4', 'пять': '5', 'шесть': '6', 'семь': '7', 'восемь': '8',
    'девять': '9', 'десять': '10', 'двенадцать': '12', 'пятнадцать': '15',
    'двадцать': '20', 'тридцать шесть': '36', 'тридцать': '30', 'сорок': '40',
    'пятьдесят': '50', 'семьдесят пять': '75', 'восемьдесят': '80', 'сто два': '102',
    'сто': '100', 'двести тридцать семь': '237', 'двести': '200', 'триста': '300',
    'трехсот': '300', 'трёхсот': '300', 'тысяча четыреста двадцать': '1420',
    'тысячу': '1000', 'две тысячи': '2000', 'девяноста пяти': '95', 'девяноста': '90',
    'пяти': '5', 'восьми': '8', 'двенадцати': '12', 'пятнадцати': '15',
}

def normalize_text(text: str) -> str:
    """Normalize text before metric calculation."""
    normalized = text.lower().replace('ё', 'е')
    normalized = normalized.replace('\r', ' ').replace('\n', ' ')
    normalized = re.sub(r'\\[nrt]', ' ', normalized)
    normalized = re.sub(r'[‐‑‒–—−]', '-', normalized)
    normalized = re.sub(r'(?<=\d)[,.](?=\d)', '<decimal>', normalized)
    normalized = re.sub(r'[«»“”„\"\'()\[\]{}]', '', normalized)
    normalized = re.sub(r'(?<=[a-zа-я])-(?=[a-zа-я])', ' ', normalized)
    normalized = re.sub(r'(?<!\w)-(?!\w)', ' ', normalized)
    normalized = re.sub(r'[,.!?;:]', '', normalized)
    for word, number in sorted(WORD_TO_NUM.items(), key=lambda item: -len(item[0])):
        normalized = re.sub(rf'\b{re.escape(word)}\b', number, normalized)
    normalized = normalized.replace('<decimal>', '.')
    normalized = re.sub(r'\bпроцентов\b|\bпроцента\b|\bпроцент\b', '%', normalized)
    normalized = re.sub(r'(\d)\s*%', r'\1%', normalized)
    normalized = re.sub(r'\bномер\b\s+', '', normalized)
    normalized = re.sub(r'№\s*', '', normalized)
    return re.sub(r'\s+', ' ', normalized).strip()

def compute_cer(reference: str, hypothesis: str) -> float:
    """Compute character error rate without extra CER dependencies."""
    ref_chars = list(reference.replace(' ', ''))
    hyp_chars = list(hypothesis.replace(' ', ''))
    if not ref_chars:
        return 0.0
    dp = list(range(len(hyp_chars) + 1))
    for ref_index, ref_char in enumerate(ref_chars, start=1):
        new_dp = [ref_index] + [0] * len(hyp_chars)
        for hyp_index, hyp_char in enumerate(hyp_chars, start=1):
            if ref_char == hyp_char:
                new_dp[hyp_index] = dp[hyp_index - 1]
            else:
                new_dp[hyp_index] = 1 + min(dp[hyp_index], new_dp[hyp_index - 1], dp[hyp_index - 1])
        dp = new_dp
    return dp[-1] / len(ref_chars)

def get_abbrev_stats(norm_ref: str, norm_hyp: str) -> tuple[int, int]:
    """Count recalled glossary abbreviations in a prediction."""
    reference_words = norm_ref.split()
    hypothesis_words = norm_hyp.split()
    found = 0
    total = 0
    for word in reference_words:
        if word in GLOSSARY_SET:
            total += 1
            if word in hypothesis_words:
                hypothesis_words.remove(word)
                found += 1
    return found, total

def collect_json_dataset_items(dataset_dir: Path) -> list[tuple[str, Path, str]]:
    """Collect items from datasets that use references.json and audio/."""
    references_path = dataset_dir / 'references.json'
    references = json.loads(references_path.read_text(encoding='utf-8'))
    items = []
    for audio_id, ref_info in references.items():
        audio_path = None
        for ext in ('.wav', '.mp3'):
            candidate = dataset_dir / 'audio' / f'{audio_id}{ext}'
            if candidate.exists():
                audio_path = candidate
                break
        if audio_path is None:
            continue
        reference_text = ref_info.get('text', '').strip()
        if reference_text:
            items.append((audio_id, audio_path, reference_text))
    return items

def collect_test_dataset_items(dataset_dir: Path) -> list[tuple[str, Path, str]]:
    """Collect items from test_dataset_v1 speech/text structure."""
    items = []
    for text_path in sorted((dataset_dir / 'text').glob('text_*.txt'), key=lambda path: int(path.stem.split('_')[-1])):
        sample_id = text_path.stem.split('_')[-1]
        audio_path = dataset_dir / 'speech' / f'speech_{sample_id}.wav'
        if not audio_path.exists():
            continue
        reference_text = text_path.read_text(encoding='utf-8').strip()
        if reference_text:
            items.append((sample_id, audio_path, reference_text))
    return items

def collect_dataset_items(dataset_dir: Path) -> list[tuple[str, Path, str]]:
    """Dispatch item collection based on the dataset format."""
    if (dataset_dir / 'references.json').exists():
        return collect_json_dataset_items(dataset_dir)
    if (dataset_dir / 'speech').is_dir() and (dataset_dir / 'text').is_dir():
        return collect_test_dataset_items(dataset_dir)
    raise FileNotFoundError(f'Unsupported dataset format: {dataset_dir}')

def build_pipeline() -> Any:
    """Build a GigaAM ASR pipeline on the best available device."""
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, trust_remote_code=True)
    if torch.cuda.is_available():
        model = model.to('cuda')
        device = 0
    else:
        model = model.to('cpu')
        device = -1
    model.eval()
    return pipeline(
        'automatic-speech-recognition',
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=device,
    )

def run_benchmark(dataset_name: str, dataset_dir: Path, asr_pipe: Any) -> dict[str, Any]:
    """Run GigaAM benchmark for a single dataset."""
    items = collect_dataset_items(dataset_dir)
    results = []
    all_refs = []
    all_hyps = []
    total_time = 0.0
    print(f'Running {dataset_name}: {len(items)} files')
    for sample_id, audio_path, reference_text in items:
        start_time = time.time()
        with torch.no_grad():
            output = asr_pipe(str(audio_path))
        hypothesis_text = output['text'].strip()
        inference_time = time.time() - start_time
        total_time += inference_time
        norm_ref = normalize_text(reference_text)
        norm_hyp = normalize_text(hypothesis_text)
        wer_score = jiwer.wer(norm_ref, norm_hyp) if norm_ref else 0.0
        cer_score = compute_cer(norm_ref, norm_hyp)
        abbr_found, abbr_total = get_abbrev_stats(norm_ref, norm_hyp)
        all_refs.append(norm_ref)
        all_hyps.append(norm_hyp)
        results.append({
            'audio_id': sample_id,
            'audio_path': str(audio_path),
            'wer': wer_score,
            'cer': cer_score,
            'abbr_found': abbr_found,
            'abbr_total': abbr_total,
            'inference_time_sec': inference_time,
            'ref_original': reference_text,
            'hyp_original': hypothesis_text,
        })
        print(f'[{dataset_name}:{sample_id}] time={inference_time:.2f}s wer={wer_score:.2%} cer={cer_score:.2%}')
    macro_wer = sum(item['wer'] for item in results) / len(results) if results else 0.0
    macro_cer = sum(item['cer'] for item in results) / len(results) if results else 0.0
    micro_wer = jiwer.wer(' '.join(all_refs), ' '.join(all_hyps)) if all_refs else 0.0
    total_char_errors = sum(compute_cer(ref, hyp) * len(ref.replace(' ', '')) for ref, hyp in zip(all_refs, all_hyps))
    total_ref_chars = sum(len(ref.replace(' ', '')) for ref in all_refs)
    total_abbr_found = sum(item['abbr_found'] for item in results)
    total_abbr_target = sum(item['abbr_total'] for item in results)
    summary = {
        'model': MODEL_ID,
        'dataset_name': dataset_name,
        'dataset_dir': str(dataset_dir),
        'wer_macro': macro_wer,
        'wer_micro': micro_wer,
        'cer_macro': macro_cer,
        'cer_micro': total_char_errors / total_ref_chars if total_ref_chars else 0.0,
        'abbr_recall': total_abbr_found / total_abbr_target if total_abbr_target else None,
        'abbr_found': total_abbr_found,
        'abbr_total': total_abbr_target,
        'avg_time_sec': total_time / len(results) if results else 0.0,
        'total_files': len(results),
        'details': results,
    }
    output_dir = RESULTS_ROOT / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / 'results_gigaam_stock.json'
    output_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved results to {output_path}')
    return summary

pipe = build_pipeline()
all_summaries = {}
started_at = time.time()
for dataset_name, dataset_dir in RUNS.items():
    all_summaries[dataset_name] = run_benchmark(dataset_name, dataset_dir, pipe)
print(f'Total elapsed minutes: {(time.time() - started_at) / 60:.1f}')
all_summaries

In [ ]:
!zip -r /content/gigaam_benchmark_results_all.zip /content/gigaam_benchmark_results_all

from google.colab import files
files.download('/content/gigaam_benchmark_results_all.zip')